# Lesson 11: Retrieval-Augmented Generation (RAG) & Vector Databases

In this self-contained notebook, you will:
1. Load embedded versions of `perceptron.md`, `own_framework.md`, and `frameworks.md`.
2. Chunk documents along word boundaries (`split_text(400, 300)`).
3. Index embeddings in **`scikit-learn` `NearestNeighbors(algorithm='ball_tree')`** and **`ChromaDB`**.
4. Query the grounded RAG chatbot (`"what is a perceptron?"`) and compute **Mean Average Precision (MAP)**.

In [ ]:
!pip install -q -U google-genai chromadb sentence-transformers scikit-learn pandas

In [ ]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import average_precision_score
from sentence_transformers import SentenceTransformer
import chromadb

docs = [
    {"source": "perceptron.md", "text": "A perceptron is a single-layer neural network unit that computes a weighted sum of inputs plus bias and applies an activation function for binary classification. Rosenblatt introduced the perceptron in 1957."},
    {"source": "own_framework.md", "text": "Multi-layer perceptrons stack linear layers with non-linear activations like ReLU and train via backpropagation and gradient descent to solve non-linear tasks."},
    {"source": "frameworks.md", "text": "Modern deep learning frameworks like PyTorch provide automatic differentiation (autograd), GPU acceleration, and modular nn.Module layers for training neural networks at scale."},
]

def split_text(text, max_length=400, min_length=120):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + max_length, len(text))
        if end < len(text) and end - start > min_length:
            sp = text.rfind(" ", start + min_length, end)
            if sp != -1:
                end = sp
        chunks.append(text[start:end].strip())
        start = end + 1
    return chunks

df = pd.DataFrame(docs)
df["chunks"] = df["text"].apply(split_text)
df = df.explode("chunks").reset_index(drop=True)

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(df["chunks"].tolist())
nbrs = NearestNeighbors(n_neighbors=2, algorithm="ball_tree").fit(embeddings)

# ChromaDB collection
chroma = chromadb.Client()
col = chroma.create_collection("nn_docs")
col.add(documents=df["chunks"].tolist(), ids=[f"doc_{i}" for i in range(len(df))])

q_vec = embedder.encode(["what is a perceptron?"])
dists, idxs = nbrs.kneighbors(q_vec)
print("Top BallTree match:", df.iloc[idxs[0][0]]["chunks"])
print("Top ChromaDB match:", col.query(query_texts=["what is a perceptron?"], n_results=1)["documents"][0][0])

In [ ]:
# MAP evaluation across relevant vs irrelevant ground truth
y_true = [1, 1, 0]
y_scores = [0.91, 0.78, 0.22]
print(f"Average Precision Score: {average_precision_score(y_true, y_scores):.3f}")